In [1]:
import pandas as pd
import numpy as np
import scanpy as sc
import scanpy.external as sce
import tqdm as tqdm
import scallop as sl
import sys
sys.path.append('C:/Users/yang2/decibel')
import module.decibel as dcb
from tqdm.notebook import tqdm
from scipy import sparse
from scipy.spatial.distance import euclidean

# 1. TMS bone marrow cells

In [2]:
adata_TMS_marrow = sc.read_h5ad("S:/Penn Dropbox/Eddie Yang/Aging/Results/TMS/TMS_marrow/TMS_marrow.h5ad")

# Preprocessing
sc.pp.normalize_total(adata_TMS_marrow, target_sum=1e4)
sc.pp.log1p(adata_TMS_marrow)

In [3]:
# Method 1: Mean Euclidean distance to cell type average based on the whole transcriptome
adata_TMS_marrow.obs.rename(columns={'celltype': 'cell_type'}, inplace=True)
adata_TMS_marrow = dcb.distance_to_celltype_mean(adata_TMS_marrow, 'age')

 67%|████████████████████████████████████████████████████████                            | 4/6 [00:01<00:00,  2.55it/s]C:\Users/yang2/decibel\module\decibel.py:151: RuntimeWarning: Mean of empty slice.
  mean_expr.loc[:, f'{b}_{ct}'] = adata[(adata.obs[batch] == b) & (adata.obs['cell_type'] == ct), :].X.toarray().mean(axis=0)
C:\Users\yang2\anaconda3\lib\site-packages\numpy\core\_methods.py:182: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
100%|█████████████████████████████████████████████████████████████████████████████████| 66/66 [00:00<00:00, 155.36it/s]
0it [00:00, ?it/s]
100%|████████████████████████████████████████████████████████████████████████████████| 616/616 [00:15<00:00, 38.94it/s]


In [4]:
# Method 2: Mean Euclidean distance to tissue avearge based on invariant genes
if sparse.issparse(adata_TMS_marrow.X):
    X = adata_TMS_marrow.X.toarray()
else:
    X = adata_TMS_marrow.X

X[np.isnan(X)] = 0
X[np.isinf(X)] = 0
adata_TMS_marrow.X = sparse.csr_matrix(X)
adata_TMS_marrow.var['means'] = pd.Series(np.asarray(adata_TMS_marrow.X.mean(axis=0)).ravel(), index=adata_TMS_marrow.var_names)

# Sort genes according to their mean expression
gene_order = adata_TMS_marrow.var.sort_values(by='means', ascending=False).index

# Euclidean distance method as described in their methods section
bins = dcb.get_bins(adata_TMS_marrow)
least_variable = dcb.get_least_variable(adata_TMS_marrow, bins)

# Mean expression across all cells
mean_expr = np.asarray(adata_TMS_marrow[:, least_variable].X.mean(axis=0)).ravel()

adata_TMS_marrow.obs['euc_dist_tissue_invar'] = [euclidean(mean_expr, adata_TMS_marrow[i, least_variable].X.toarray().ravel()) for i in tqdm(range(adata_TMS_marrow.shape[0]))]

  0%|          | 0/40220 [00:00<?, ?it/s]

In [5]:
# Method 3: Scallop
scal_TMS_marrow = sl.Scallop(adata_TMS_marrow)
score = sl.tl.getScore(scal_TMS_marrow, res=1.2, n_trials=50, frac_cells=0.9, do_return=True)
adata_TMS_marrow.obs['scallop_score'] = score


2026-03-25 01:41:06,848 - scallop / root - WARNING - Neighbors not in annData. Computing neighbors


         Falling back to preprocessing with `sc.pp.pca` and default params.



2026-03-25 01:42:05,471 - scallop / root - INFO - Obtaining score with following parameters: 
 Resolution: 1.2
 Fraction of cells: 0.9
 Number of trials: 50
 Score type: freq
Clustering type: leiden, seed: None

2026-03-25 01:42:05,474 - scallop / root - INFO - The Bootstrap object did not exist and will been created.

2026-03-25 01:42:05,474 - scallop / root - INFO - Calculating the bootstrap matrix.

2026-03-25 01:42:07,339 - scallop / root - INFO - Obtaining bootstrap matrix. 1 processes are used at once.
100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [01:07<00:00,  1.34s/it]

2026-03-25 01:43:14,589 - scallop / root - INFO - Renaming identities.

2026-03-25 01:43:37,226 - scallop / root - INFO - Applying mapping of equivalent unknowns with threshold 0.5.

2026-03-25 01:43:37,354 - scallop / root - INFO - Calculating freqScore.
C:\Users\yang2\anaconda3\lib\site-packages\scallop\classes\bootstrap.py:366: FutureWarning: Unlike other red

In [6]:
# Save outputs
adata_TMS_marrow.obs.to_csv("adata_TMS_marrow_metadata.csv")

# 2. Liver hepatocytes

In [7]:
adata_liver = sc.read_h5ad("S:/Penn Dropbox/Eddie Yang/Aging/Data/C57BL6J_liver_hippocampus/liver_data_hep_only.h5ad")

# Preprocessing
sc.pp.normalize_total(adata_liver, target_sum=1e4)
sc.pp.log1p(adata_liver)

C:\Users\yang2\anaconda3\lib\site-packages\scanpy\preprocessing\_normalization.py:196: UserWarning: Some cells have zero counts
  warn(UserWarning('Some cells have zero counts'))
C:\Users\yang2\anaconda3\lib\site-packages\scanpy\preprocessing\_simple.py:351: RuntimeWarning: invalid value encountered in log1p
  np.log1p(X, out=X)


In [8]:
# Method 1: Mean Euclidean distance to cell type average based on the whole transcriptome
adata_liver.obs.rename(columns={'annotation': 'cell_type'}, inplace=True)
adata_liver = dcb.distance_to_celltype_mean(adata_liver, 'age')

100%|███████████████████████████████████████████████████████████████████████████| 56372/56372 [04:10<00:00, 225.18it/s]


In [9]:
# Method 2: Mean Euclidean distance to tissue avearge based on invariant genes
if sparse.issparse(adata_liver.X):
    X = adata_liver.X.toarray()
else:
    X = adata_liver.X

X[np.isnan(X)] = 0
X[np.isinf(X)] = 0
adata_liver.X = sparse.csr_matrix(X)
adata_liver.var['means'] = pd.Series(np.asarray(adata_liver.X.mean(axis=0)).ravel(), index=adata_liver.var_names)

# Sort genes according to their mean expression
gene_order = adata_liver.var.sort_values(by='means', ascending=False).index

# Euclidean distance method as described in their methods section
bins = dcb.get_bins(adata_liver)
least_variable = dcb.get_least_variable(adata_liver, bins)

# Mean expression across all cells
mean_expr = np.asarray(adata_liver[:, least_variable].X.mean(axis=0)).ravel()

adata_liver.obs['euc_dist_tissue_invar'] = [euclidean(mean_expr, adata_liver[i, least_variable].X.toarray().ravel()) for i in tqdm(range(adata_liver.shape[0]))]

  0%|          | 0/94601 [00:00<?, ?it/s]

In [10]:
# Method 3: Scallop
scal_liver = sl.Scallop(adata_liver)
score = sl.tl.getScore(scal_liver, res=1.2, n_trials=50, frac_cells=0.9, do_return=True)
adata_liver.obs['scallop_score'] = score


2026-03-25 01:52:14,716 - scallop / root - WARNING - Neighbors not in annData. Computing neighbors

2026-03-25 01:52:17,736 - scallop / root - INFO - Obtaining score with following parameters: 
 Resolution: 1.2
 Fraction of cells: 0.9
 Number of trials: 50
 Score type: freq
Clustering type: leiden, seed: None

2026-03-25 01:52:17,740 - scallop / root - INFO - The Bootstrap object did not exist and will been created.

2026-03-25 01:52:17,741 - scallop / root - INFO - Calculating the bootstrap matrix.

2026-03-25 01:52:24,424 - scallop / root - INFO - Obtaining bootstrap matrix. 1 processes are used at once.
100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [04:29<00:00,  5.40s/it]

2026-03-25 01:56:54,178 - scallop / root - INFO - Renaming identities.

2026-03-25 01:57:59,500 - scallop / root - INFO - Applying mapping of equivalent unknowns with threshold 0.5.

2026-03-25 01:57:59,549 - scallop / root - INFO - Calculating freqScore.
C:\Users

In [11]:
# Save outputs
adata_liver.obs.to_csv("adata_liver_metadata.csv")

# 3. Kidney PT cells

In [12]:
adata_kidney_PT_only = sc.read_h5ad("S:/Penn Dropbox/Eddie Yang/RAGE24/cellranger_flowcell_combine/kidney_aging_samples_integrated_PT_only.h5ad")

# Preprocessing
sc.pp.normalize_total(adata_kidney_PT_only, target_sum=1e4)
sc.pp.log1p(adata_kidney_PT_only)

C:\Users\yang2\anaconda3\lib\site-packages\scanpy\preprocessing\_normalization.py:196: UserWarning: Some cells have zero counts
  warn(UserWarning('Some cells have zero counts'))
C:\Users\yang2\anaconda3\lib\site-packages\scanpy\preprocessing\_simple.py:351: RuntimeWarning: invalid value encountered in log1p
  np.log1p(X, out=X)


In [13]:
# Method 1: Mean Euclidean distance to cell type average based on the whole transcriptome
adata_kidney_PT_only.obs.rename(columns={'celltype_refined2': 'cell_type'}, inplace=True)
adata_kidney_PT_only = dcb.distance_to_celltype_mean(adata_kidney_PT_only, 'age_wks')

100%|█████████████████████████████████████████████████████████████████████████████████| 14/14 [00:00<00:00, 143.74it/s]


In [14]:
# Method 2: Mean Euclidean distance to tissue avearge based on invariant genes
if sparse.issparse(adata_kidney_PT_only.X):
    X = adata_kidney_PT_only.X.toarray()
else:
    X = adata_kidney_PT_only.X

X[np.isnan(X)] = 0
X[np.isinf(X)] = 0
adata_kidney_PT_only.X = sparse.csr_matrix(X)
adata_kidney_PT_only.var['means'] = pd.Series(np.asarray(adata_kidney_PT_only.X.mean(axis=0)).ravel(), index=adata_kidney_PT_only.var_names)

# Sort genes according to their mean expression
gene_order = adata_kidney_PT_only.var.sort_values(by='means', ascending=False).index

# Euclidean distance method as described in their methods section
bins = dcb.get_bins(adata_kidney_PT_only)
least_variable = dcb.get_least_variable(adata_kidney_PT_only, bins)

# Mean expression across all cells
mean_expr = np.asarray(adata_kidney_PT_only[:, least_variable].X.mean(axis=0)).ravel()

adata_kidney_PT_only.obs['euc_dist_tissue_invar'] = [euclidean(mean_expr, adata_kidney_PT_only[i, least_variable].X.toarray().ravel()) for i in tqdm(range(adata_kidney_PT_only.shape[0]))]

  0%|          | 0/20873 [00:00<?, ?it/s]

In [15]:
# Method 3: Scallop
scal_kidney_PT_only = sl.Scallop(adata_kidney_PT_only)
score = sl.tl.getScore(scal_kidney_PT_only, res=1.2, n_trials=50, frac_cells=0.9, do_return=True)
adata_kidney_PT_only.obs['scallop_score'] = score


2026-03-25 01:59:54,068 - scallop / root - WARNING - Neighbors not in annData. Computing neighbors

2026-03-25 01:59:54,909 - scallop / root - INFO - Obtaining score with following parameters: 
 Resolution: 1.2
 Fraction of cells: 0.9
 Number of trials: 50
 Score type: freq
Clustering type: leiden, seed: None

2026-03-25 01:59:54,911 - scallop / root - INFO - The Bootstrap object did not exist and will been created.

2026-03-25 01:59:54,912 - scallop / root - INFO - Calculating the bootstrap matrix.

2026-03-25 01:59:56,309 - scallop / root - INFO - Obtaining bootstrap matrix. 1 processes are used at once.
100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [00:50<00:00,  1.01s/it]

2026-03-25 02:00:46,959 - scallop / root - INFO - Renaming identities.

2026-03-25 02:00:54,588 - scallop / root - INFO - Applying mapping of equivalent unknowns with threshold 0.5.

2026-03-25 02:00:54,707 - scallop / root - INFO - Calculating freqScore.
C:\Users

In [16]:
# Save outputs
adata_kidney_PT_only.obs.to_csv("adata_kidney_PT_only_metadata.csv")